In [1]:
import requests

In [2]:
import requests

BASE_URL = "https://gamma-api.polymarket.com"
HEADERS = {
    "Accept": "application/json",
}


In [5]:
import requests

BASE_URL = "https://gamma-api.polymarket.com"
HEADERS = {"Accept": "application/json"}

def get_active_events(limit=50, offset=0):
    """Fetch active events (paginated)"""
    params = {
        "order": "id",
        "ascending": "false",
        "closed": "false",
        "limit": limit,
        "offset": offset
    }
    url = f"{BASE_URL}/events"
    r = requests.get(url, headers=HEADERS, params=params, timeout=10)
    r.raise_for_status()
    return r.json()

def extract_yes_no(market: dict):
    yes_price = None
    no_price = None
    for outcome in market.get("outcomes", []):
        name = outcome.get("name", "").lower()
        price = outcome.get("price")
        if name == "yes":
            yes_price = price
        elif name == "no":
            no_price = price
    return yes_price, no_price

def parse_event(event: dict):
    parsed = []
    for market in event.get("markets", []):
        yes, no = extract_yes_no(market)
        parsed.append({
            "event_id": event["id"],
            "event_title": event["title"],
            "market_id": market["id"],
            "market_question": market["question"],
            "yes_price": yes,
            "no_price": no,
            "liquidity": market.get("liquidity"),
            "volume": market.get("volume"),
            "end_date": market.get("end_date"),
            "slug": market.get("slug")
        })
    return parsed

# Fetch all active markets
all_markets = []
offset = 0
limit = 50

while True:
    events = get_active_events(limit=limit, offset=offset)
    if not events:
        break
    for event in events:
        all_markets.extend(parse_event(event))
    offset += limit  # Move to next page

print(f"Total live markets fetched: {len(all_markets)}")
print(all_markets[0])  # Example


AttributeError: 'str' object has no attribute 'get'

In [6]:
events = get_active_events(limit=limit, offset=offset)

In [9]:
events[0]

{'id': '155214',
 'ticker': 'btc-updown-5m-1768133400',
 'slug': 'btc-updown-5m-1768133400',
 'title': 'Bitcoin Up or Down - January 11, 7:10AM-7:15AM ET',
 'description': 'This market will resolve to "Up" if the Bitcoin price at the end of the time range specified in the title is greater than or equal to the price at the beginning of that range. Otherwise, it will resolve to "Down".\nThe resolution source for this market is information from Chainlink, specifically the BTC/USD data stream available at https://data.chain.link/streams/btc-usd.\nPlease note that this market is about the price according to Chainlink data stream BTC/USD, not according to other sources or spot markets.',
 'resolutionSource': 'https://data.chain.link/streams/btc-usd',
 'startDate': '2026-01-10T12:17:06.820057Z',
 'creationDate': '2026-01-10T12:17:06.820052Z',
 'endDate': '2026-01-11T12:15:00Z',
 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png',
 'icon': 'https://polymarket-uplo

In [10]:
import pandas as pd
import ast  # To safely parse outcomes if they are JSON-like strings

# Sample: events[0] (from your data)
event = events[0]  # Replace this with API fetch results

rows = []

for market in event.get("markets", []):
    # Basic info
    title = market.get("question") or event.get("title")
    description = market.get("description") or event.get("description")
    start_date = market.get("startDate") or event.get("startDate")
    end_date = market.get("endDate") or event.get("endDate")
    volume = float(market.get("volume") or 0)

    # Parse outcomes (could be string or list)
    outcomes_raw = market.get("outcomes")
    if isinstance(outcomes_raw, str):
        try:
            outcomes = ast.literal_eval(outcomes_raw)  # Convert '["Up","Down"]' → list
        except:
            outcomes = [outcomes_raw]
    elif isinstance(outcomes_raw, list):
        outcomes = outcomes_raw
    else:
        outcomes = []

    # Parse prices if available (sometimes yes/no only)
    # Polymarket often uses "price" per outcome, here we default None
    # For yes/no markets, we map accordingly
    yes_price = None
    no_price = None
    if len(outcomes) == 2:
        name0 = outcomes[0].lower()
        name1 = outcomes[1].lower()
        # We assume a YES/NO style
        if "yes" in name0 or "up" in name0:
            yes_price = market.get("priceYes") or None
            no_price = market.get("priceNo") or None
        elif "yes" in name1 or "up" in name1:
            yes_price = market.get("priceNo") or None
            no_price = market.get("priceYes") or None
    elif len(outcomes) > 2:
        # For multiple-choice markets, create a row per outcome
        for outcome in outcomes:
            rows.append({
                "title": title,
                "description": description,
                "start_date": start_date,
                "end_date": end_date,
                "volume": volume,
                "outcome": outcome,
                "yes_price": None,
                "no_price": None
            })
        continue  # Skip the generic yes/no row

    # Append normal yes/no row
    rows.append({
        "title": title,
        "description": description,
        "start_date": start_date,
        "end_date": end_date,
        "volume": volume,
        "outcome": "yes/no",
        "yes_price": yes_price,
        "no_price": no_price
    })

# Convert to Pandas DataFrame
df = pd.DataFrame(rows)

print(df)


                                               title  \
0  Bitcoin Up or Down - January 11, 7:10AM-7:15AM ET   

                                         description            start_date  \
0  This market will resolve to "Up" if the Bitcoi...  2026-01-10T12:16:51Z   

               end_date  volume outcome yes_price no_price  
0  2026-01-11T12:15:00Z     0.0  yes/no      None     None  


In [13]:
df.head()

,title,description,start_date,end_date,volume,outcome,yes_price,no_price
0,"Bitcoin Up or Down - January 11, 7:10AM-7:15AM ET","This market will resolve to ""Up"" if the Bitcoi...",2026-01-10T12:16:51Z,2026-01-11T12:15:00Z,0.0,yes/no,None,None
